In [19]:
import requests
import json
import re

In [3]:
BASE_URL = "http://2cheapbv.picqer.com"
USERNAME = "webshopimporter"
PASSWORD = "wdbT4zIUhQ@"
API_KEY = "xhDM2BVsiMzgcUkJAX08RpymByC1oQF1zreynUOMPxUX7Vc9"

In [4]:
with open("lightspeed_variants.json", "r") as f:
    all_variants = json.load(f)

In [14]:
print(json.dumps(all_variants[1000], indent=2))

{
  "id": 319155756,
  "createdAt": "2026-01-06T14:09:16+01:00",
  "updatedAt": "2026-01-07T01:15:23+01:00",
  "isDefault": true,
  "sortOrder": 1,
  "articleCode": "CYT-0578",
  "ean": "8718347831189",
  "sku": "24327039331",
  "hs": null,
  "unitPrice": 0,
  "unitUnit": null,
  "priceExcl": 39.6281,
  "priceIncl": 47.95,
  "priceCost": 17.46,
  "oldPriceExcl": 0,
  "oldPriceIncl": 0,
  "stockTracking": "enabled",
  "stockLevel": 0,
  "stockAlert": 0,
  "stockMinimum": 0,
  "stockSold": 0,
  "stockBuyMininum": 1,
  "stockBuyMinimum": 1,
  "stockBuyMaximum": 10000,
  "weight": 25,
  "weightValue": "25.000",
  "weightUnit": "g",
  "volume": 0,
  "volumeValue": 0,
  "volumeUnit": "ml",
  "colli": 0,
  "sizeX": 0,
  "sizeY": 0,
  "sizeZ": 0,
  "sizeXValue": null,
  "sizeYValue": null,
  "sizeZValue": null,
  "sizeUnit": "cm",
  "matrix": false,
  "title": "Default",
  "taxType": null,
  "image": false,
  "tax": {
    "resource": {
      "id": 48606,
      "url": "taxes/48606",
      "link

In [11]:
# Get picqer productfields
response = requests.get(f"{BASE_URL}/api/v1/productfields", auth=(API_KEY, ""))
product_fields = response.json()
print(json.dumps(product_fields, indent=2))

[
  {
    "idproductfield": 360,
    "title": "Beschikbaar",
    "type": "select",
    "values": [
      "LEVERTIJD",
      ""
    ],
    "required": false,
    "visible_picklist": false,
    "visible_invoice": false,
    "visible_shippinglist": false,
    "visible_portal": false,
    "visible_purchase_order": true,
    "visible_receipt": true
  },
  {
    "idproductfield": 5745,
    "title": "Beschikbaar - Aangemaakt op",
    "type": "text",
    "values": [],
    "required": false,
    "visible_picklist": false,
    "visible_invoice": false,
    "visible_shippinglist": false,
    "visible_portal": true,
    "visible_purchase_order": true,
    "visible_receipt": false
  },
  {
    "idproductfield": 253,
    "title": "Opmerking Inkoop",
    "type": "text",
    "values": [],
    "required": false,
    "visible_picklist": false,
    "visible_invoice": false,
    "visible_shippinglist": false,
    "visible_portal": false,
    "visible_purchase_order": true,
    "visible_receipt": true
  },

In [17]:
# Get picqer tags
response = requests.get(f"{BASE_URL}/api/v1/tags", auth=(API_KEY, ""))
tags = response.json()
print(json.dumps(tags, indent=2))

[
  {
    "idtag": 8666,
    "title": "2Cheap",
    "color": "#43c059",
    "inherit": true,
    "textColor": "#000000"
  },
  {
    "idtag": 64272,
    "title": "AFHAAL",
    "color": "#f000cf",
    "inherit": false,
    "textColor": "#000000"
  },
  {
    "idtag": 25613,
    "title": "Bol BE",
    "color": "#d0d0dd",
    "inherit": false,
    "textColor": "#000000"
  },
  {
    "idtag": 27013,
    "title": "Bol NL",
    "color": "#2929a3",
    "inherit": true,
    "textColor": "#FFFFFF"
  },
  {
    "idtag": 8125,
    "title": "Briefpost",
    "color": "#1aa8b0",
    "inherit": true,
    "textColor": "#000000"
  },
  {
    "idtag": 11964,
    "title": "DHL Small",
    "color": "#0000f0",
    "inherit": true,
    "textColor": "#FFFFFF"
  },
  {
    "idtag": 18778,
    "title": "DHL XXL",
    "color": "#2e8b82",
    "inherit": true,
    "textColor": "#FFFFFF"
  },
  {
    "idtag": 17307,
    "title": "DPD",
    "color": "#e5f000",
    "inherit": false,
    "textColor": "#000000"
  },
 

In [ ]:
send_options_field = [product_field for product_field in product_fields if product_field["title"] == "Verzend"][0]
available_created_at_field = [product_field for product_field in product_fields if product_field["name"] == "Beschikbaar - Aangemaakt op"][0]

In [ ]:
def get_lightspeed_product_weight(variant):
    weight_unit = variant["weightUnit"]

    if weight_unit == "g":
        return variant["weight"] 
    
    if weight_unit == "kg":
        return variant["weight"] * 1000
    
    print(f"Unknown weight unit: {weight_unit}")
    return variant["weight"]

In [15]:
def get_sending_option(weight):
    if weight < 25:
        return "Briefpost"
    elif weight < 1001:
        return "DHL Small"
    elif weight < 10000:
        return "DPD"
    elif weight < 50000:
        return "DHL XXL"
    elif weight >= 50000:
        return "Speciaal Transport"
    else:
        return "VERZENDTYPE ONBEKEND"

SyntaxError: incomplete input (326984710.py, line 5)

In [ ]:
def parse_delivery_days(data01: str) -> int | None:
    """Extract the first number from a Lightspeed data01 delivery string, e.g. '3-5 dagen' → 3"""
    if not data01:
        return None
    match = re.search(r'\d+', data01)
    return int(match.group()) if match else None

In [ ]:


put_requests = []

while len(all_variants) > 0:
    variant = all_variants.pop()

    url = f"{BASE_URL}/api/v1/products?productcode={variant['sku']}"
    response = requests.get(url, auth=(API_KEY, ""))
    product = response.json()

    print(json.dumps(product, indent=2))

    product_weight = get_lightspeed_product_weight(variant)
    product_sending_option = get_sending_option(product_weight)
    product_tag = next((tag for tag in tags if tag["name"] == product_sending_option), None)

    if not product_tag:
        print(f"Tag not found for sending option: {product_sending_option}")
        continue

    delivery_days = parse_delivery_days(variant.get("data01", ""))
    delivery_time_text = variant.get("data01", "")

    put_requests.append({
        "id": product["idproduct"],
        "payload": {
            "weight": product_weight,
            "deliverytime": delivery_days,  # built-in Picqer field (integer days)
            "tag": {
                product_sending_option: product_tag
            },
            "productfields": [
                {
                    "idproductfield": send_options_field["idproductfield"],
                    "value": product_sending_option
                },
            ]
        }
    })


[
  {
    "idproduct": 1822517,
    "idvatgroup": 3583,
    "idsupplier": 11409,
    "productcode": "24327000007",
    "name": "Hofftech Bankhamer - Glasfiber Steel - 500 gram - 35 cm - Ergonomisch Handvat",
    "price": 8.47,
    "fixedstockprice": 3.99,
    "productcode_supplier": "BNS-006132",
    "deliverytime": 5,
    "description": "Zie Omschrijving Webshop",
    "barcode": "8717729114827",
    "unlimitedstock": false,
    "assembled": false,
    "type": "normal",
    "weight": 2,
    "length": null,
    "width": null,
    "height": null,
    "minimum_purchase_quantity": 1,
    "purchase_in_quantities_of": 1,
    "hs_code": null,
    "country_of_origin": null,
    "show_on_portal": true,
    "active": true,
    "created": "2015-12-30 13:22:03",
    "updated": "2026-02-25 00:22:42",
    "comment_count": 0,
    "analysis_abc_classification": "C",
    "analysis_pick_amount_per_day": "0.143",
    "tags": {
      "Briefpost": {
        "idtag": 8125,
        "title": "Briefpost",
    

NameError: name 'get_lightspeed_product_weight' is not defined